In [ ]:
!pip install torch_geometric

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data

# ==========================================
# 1. LOAD AND CLEAN DATA
# ==========================================
df = pd.read_csv("wustl-ehms-2020_with_attacks_categories.csv")
drop_cols = ['Dir', 'SrcAddr', 'DstAddr', 'SrcMac', 'DstMac', 'Sport', 'Dport', 'Label', 'Packet_num']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

if 'Flgs' in df.columns:
    df['Flgs'] = LabelEncoder().fit_transform(df['Flgs'].astype(str))

target_enc = LabelEncoder()
y = target_enc.fit_transform(df['Attack Category'])

X_df = df.drop(columns=['Attack Category'], errors='ignore')
X_df = X_df.select_dtypes(include=[np.number]).fillna(0)
X = X_df.values
feats = X_df.columns.tolist()

# ==========================================
# 2. SPLITS AND WEIGHTS
# ==========================================
indices = np.arange(len(df))
idx_train, idx_temp, y_train, y_temp = train_test_split(indices, y, test_size=0.3, stratify=y, random_state=42)
idx_val, idx_test, y_val, y_test = train_test_split(idx_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)

# ==========================================
# 3. SCALING
# ==========================================
scaler = StandardScaler()
scaler.fit(X[idx_train])
X_scaled = scaler.transform(X)

# ==========================================
# 4. RANDOM TEST SAMPLE
# ==========================================
normal_id = list(target_enc.classes_).index('normal')
attack_ids_in_test = np.where(y_test != normal_id)[0]
random_test_idx = np.random.choice(attack_ids_in_test)
global_idx = idx_test[random_test_idx]
true_label_id = y_test[random_test_idx]

# ==========================================
# 5. GRAPH CONSTRUCTION
# ==========================================
print("Computing Network Topologies (Graph Edges)...")
num_nodes = len(X_scaled)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[idx_train] = True
val_mask[idx_val] = True
test_mask[idx_test] = True

# Temporal Edges
source_nodes = np.arange(num_nodes - 1)
target_nodes = np.arange(1, num_nodes)
temporal_edge_index = np.concatenate([np.vstack((source_nodes, target_nodes)), np.vstack((target_nodes, source_nodes))], axis=1)

# KNN Edges
k = 5
knn = NearestNeighbors(n_neighbors=k+1, metric='cosine', n_jobs=-1)
knn.fit(X_scaled)
distances, knn_indices = knn.kneighbors(X_scaled)

knn_sources = np.repeat(np.arange(num_nodes), k)
knn_targets = knn_indices[:, 1:].flatten()
knn_edge_index = np.concatenate([np.vstack((knn_sources, knn_targets)), np.vstack((knn_targets, knn_sources))], axis=1)

combined_edges = np.concatenate([temporal_edge_index, knn_edge_index], axis=1)
edge_index_unique = np.unique(combined_edges, axis=1)

# CREATE THE 'data' OBJECT
data = Data(
    x=torch.tensor(X_scaled, dtype=torch.float),
    edge_index=torch.tensor(edge_index_unique, dtype=torch.long),
    y=torch.tensor(y, dtype=torch.long)
)
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"Graph Data Ready! Nodes: {data.num_nodes}, Edges: {data.num_edges}")

Computing Network Topologies (Graph Edges)...
Graph Data Ready! Nodes: 16318, Edges: 128208


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class Hybrid_CNNLSTM_GAT(nn.Module):
    def __init__(self, num_classes):
        super(Hybrid_CNNLSTM_GAT, self).__init__()

        # --- CNN Branch ---
        self.conv = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=3, padding=1)
        self.cnn_attn_dense = nn.Linear(128, 1)
        self.cnn_proj = nn.Linear(128, 42)

        # --- LSTM Branch ---
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, batch_first=True)
        self.lstm_attn_dense = nn.Linear(64, 1)
        self.lstm_proj = nn.Linear(64, 42)

        # --- Fusion ---
        self.batch_norm = nn.BatchNorm1d(126)
        self.dropout_cnn = nn.Dropout(p=0.1105)

        # --- GAT_v2 Branch ---
        # The input to GAT is the 126-D fused embeddings!
        self.gat1 = GATConv(126, 64, heads=4, dropout=0.21001645973261923)
        self.bn1 = torch.nn.BatchNorm1d(64 * 4)

        self.gat2 = GATConv(64 * 4, 32, heads=4, dropout=0.21001645973261923)
        self.bn2 = torch.nn.BatchNorm1d(32 * 4)

        self.gat3 = GATConv(32 * 4, num_classes, heads=1, concat=False, dropout=0.21001645973261923)
        self.dropout_gat = nn.Dropout(p=0.21001645973261923)

    def forward(self, x_seq, edge_index):
        # 1. CNN Extraction
        x_cnn = x_seq.transpose(1, 2)
        cnn_b = F.relu(self.conv(x_cnn)).transpose(1, 2)
        cnn_a = F.softmax(torch.tanh(self.cnn_attn_dense(cnn_b)), dim=1)
        cnn_out = F.relu(self.cnn_proj(torch.mean(cnn_b * cnn_a, dim=1)))

        # 2. LSTM Extraction
        lstm_b, _ = self.lstm(x_seq)
        lstm_a = F.softmax(torch.tanh(self.lstm_attn_dense(lstm_b)), dim=1)
        lstm_out = F.relu(self.lstm_proj(torch.mean(lstm_b * lstm_a, dim=1)))

        # 3. Fusion -> Node Embeddings
        fused = cnn_out * lstm_out
        node_embeddings = torch.cat([cnn_out, lstm_out, fused], dim=1)
        node_embeddings = self.dropout_cnn(self.batch_norm(node_embeddings))

        # 4. GAT_v2 Message Passing
        x_graph = self.gat1(node_embeddings, edge_index)
        x_graph = self.bn1(x_graph)
        x_graph = F.elu(x_graph)
        x_graph = self.dropout_gat(x_graph)

        x_graph = self.gat2(x_graph, edge_index)
        x_graph = self.bn2(x_graph)
        x_graph = F.elu(x_graph)
        x_graph = self.dropout_gat(x_graph)

        out = self.gat3(x_graph, edge_index)
        return out

print("Hybrid CNN+LSTM+GAT_v2 Architecture Defined!")

Hybrid CNN+LSTM+GAT_v2 Architecture Defined!


In [ ]:
import torch.optim as optim
import copy
import torch

print("--- Initializing Ultimate Hybrid Model (GAT_v2) ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware Activated: {device}")

data.x_3d = data.x.unsqueeze(-1)

data = data.to(device)

hybrid_model = Hybrid_CNNLSTM_GAT(num_classes=len(target_enc.classes_)).to(device)

optimizer = optim.Adam(hybrid_model.parameters(), lr=0.009640661422823864, weight_decay=2.5317495991481588e-05)
criterion = torch.nn.CrossEntropyLoss()

best_val_acc = 0
best_model_state = None
patience_counter = 0

print("🔥 Training End-to-End Hybrid Model on GPU 🔥")
for epoch in range(1, 2001):
    # Train
    hybrid_model.train()
    optimizer.zero_grad()
    out = hybrid_model(data.x_3d, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    # Evaluate
    hybrid_model.eval()
    with torch.no_grad():
        out = hybrid_model(data.x_3d, data.edge_index)
        pred = out.argmax(dim=1)
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(hybrid_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 50 == 0:
        print(f'Epoch: {epoch:03d} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}')

    if patience_counter >= 200:
        print(f"\nEarly stopping at epoch {epoch}!")
        break

if best_model_state is not None:
    hybrid_model.load_state_dict(best_model_state)

--- Initializing Ultimate Hybrid Model (GAT_v2) ---
Hardware Activated: cuda
🔥 Training End-to-End Hybrid Model on GPU 🔥
Epoch: 050 | Loss: 0.2390 | Val Acc: 0.8791
Epoch: 100 | Loss: 0.1962 | Val Acc: 0.9359
Epoch: 150 | Loss: 0.1884 | Val Acc: 0.9363
Epoch: 200 | Loss: 0.1779 | Val Acc: 0.0584
Epoch: 250 | Loss: 0.1691 | Val Acc: 0.0711
Epoch: 300 | Loss: 0.1685 | Val Acc: 0.0690
Epoch: 350 | Loss: 0.1720 | Val Acc: 0.6654

Early stopping at epoch 365!


In [ ]:
from sklearn.metrics import classification_report
import torch.nn.functional as F
import numpy as np
import torch

hybrid_model.eval()
with torch.no_grad():
    out = hybrid_model(data.x_3d, data.edge_index)
    test_logits = out[data.test_mask]
    test_preds = test_logits.argmax(dim=1).cpu().numpy()
    test_true = data.y[data.test_mask].cpu().numpy()

final_acc = (test_preds == test_true).sum() / len(test_true)
print(f"\n🏆 ULTIMATE HYBRID (APPROACH 3) FINAL TEST ACCURACY: {final_acc * 100:.2f}% 🏆\n")

target_names = [str(c) for c in target_enc.classes_]
print("Classification Report:")
print(classification_report(test_true, test_preds, target_names=target_names))

test_indices = np.where(data.test_mask.cpu().numpy())[0]
target_local_idx = np.where(test_indices == global_idx)[0][0]

sample_logits = test_logits[target_local_idx]
true_label_id = test_true[target_local_idx]

sample_probs = F.softmax(sample_logits, dim=0).cpu().numpy()
hybrid_pred = np.argmax(sample_probs)

print("\n" + "="*50)
print(f" APPROACH 3 (HYBRID GAT_v2) BREAKDOWN FOR TEST SAMPLE {global_idx}")
print("="*50)
print(f"Hybrid Probabilities  : {np.round(sample_probs, 4)}")
print("-" * 50)
print(f"Final Prediction      : {target_enc.classes_[hybrid_pred].upper()}")
print(f"Truth                 : {target_enc.classes_[true_label_id].upper()}")
print("="*50)


🏆 ULTIMATE HYBRID (APPROACH 3) FINAL TEST ACCURACY: 94.40% 🏆

Classification Report:
                 precision    recall  f1-score   support

Data Alteration       1.00      1.00      1.00       139
       Spoofing       0.76      0.27      0.40       168
         normal       0.95      0.99      0.97      2141

       accuracy                           0.94      2448
      macro avg       0.90      0.75      0.79      2448
   weighted avg       0.94      0.94      0.93      2448


 APPROACH 3 (HYBRID GAT_v2) BREAKDOWN FOR TEST SAMPLE 4770
Hybrid Probabilities  : [9.999e-01 0.000e+00 1.000e-04]
--------------------------------------------------
Final Prediction      : DATA ALTERATION
Truth                 : DATA ALTERATION
